# Assignment 7: Creating and Fixing Mode Collapse in a GAN

**Pipeline**
1. Healthy DCGAN baseline (reference point for the metrics)
2. Broken DCGAN: no BatchNorm in G + tiny G + high learning rate
3. Fixed DCGAN: **one** technique applied (selected in the config cell)
4. Side-by-side comparison with quantitative diversity metrics
5. Written explanation (fill in after you see your own results)

**Assumption:** Assignment 4's DCGAN is not available to me, so this is a standard DCGAN on MNIST at 32x32. If your A4 model uses a different dataset or architecture, replace `make_generator`, `make_discriminator` and `get_loader` only. Everything else is model-agnostic.

**Outputs saved to the working directory (your screenshots):** `baseline_output.png`, `collapsed_output.png`, `improved_output.png`, `comparison.png`

In [ ]:
import random, time
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision, torchvision.transforms as T
import matplotlib.pyplot as plt
from torchvision.utils import make_grid

SEED = 42
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed()

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)
if device.type == "cpu":
    print("WARNING: CPU only. 20 epochs x 3 runs will be slow. Use a GPU runtime (Colab: Runtime > Change runtime type > T4).")

## 1. Configuration
All experiment knobs live here. Nothing else needs editing to run the assignment.

In [ ]:
Z_DIM, IMG_SIZE, BATCH = 100, 32, 128
EPOCHS = 20                      # assignment asks for ~15-20

CONFIGS = {
    # healthy reference
    "baseline": dict(ngf=64, g_bn=True,  lr_g=2e-4, lr_d=2e-4, real_label=1.0),

    # deliberately broken: all three assignment-listed breakages combined
    #   - ngf=8    : generator capacity cut 8x
    #   - g_bn=False: BatchNorm removed from generator
    #   - lr=1e-2  : ~50x the standard 2e-4
    "broken":   dict(ngf=8,  g_bn=False, lr_g=1e-2, lr_d=1e-2, real_label=1.0),
}

# ---- the ONE fix. Choose 'ttur' or 'label_smoothing'. Do not combine them: you need to attribute the effect.
FIX = "ttur"
if FIX == "ttur":
    # Different learning rates for G and D, at sane magnitudes.
    # NOTE: this also lowers the LR from 1e-2. See section 6: that confound is real and you should address it in your write-up.
    CONFIGS["fixed"] = dict(ngf=8, g_bn=False, lr_g=1e-4, lr_d=4e-4, real_label=1.0)
elif FIX == "label_smoothing":
    # One-sided label smoothing (real=0.9), everything else identical to 'broken'.
    # This is the cleanest ablation, but with lr=1e-2 it may not rescue training.
    CONFIGS["fixed"] = dict(ngf=8, g_bn=False, lr_g=1e-2, lr_d=1e-2, real_label=0.9)
else:
    raise ValueError(FIX)

for k, v in CONFIGS.items(): print(k, v)

## 2. Data

In [ ]:
def get_loader():
    tf = T.Compose([T.Resize(IMG_SIZE), T.ToTensor(), T.Normalize((0.5,), (0.5,))])  # -> [-1, 1] for tanh
    ds = torchvision.datasets.MNIST("./data", train=True, download=True, transform=tf)
    return torch.utils.data.DataLoader(ds, batch_size=BATCH, shuffle=True, drop_last=True,
                                       num_workers=2, pin_memory=(device.type == "cuda"))

loader = get_loader()
real_batch, _ = next(iter(loader))
print(real_batch.shape)
plt.figure(figsize=(5,5)); plt.axis("off"); plt.title("Real samples")
plt.imshow(make_grid(real_batch[:64], nrow=8, normalize=True, value_range=(-1,1)).permute(1,2,0).cpu()); plt.show()

## 3. Models

In [ ]:
def make_generator(ngf=64, use_bn=True):
    def block(i, o, k, s, p):
        layers = [nn.ConvTranspose2d(i, o, k, s, p, bias=not use_bn)]
        if use_bn: layers.append(nn.BatchNorm2d(o))
        layers.append(nn.ReLU(True))
        return layers
    return nn.Sequential(
        *block(Z_DIM, ngf*4, 4, 1, 0),      # 1  -> 4
        *block(ngf*4, ngf*2, 4, 2, 1),      # 4  -> 8
        *block(ngf*2, ngf,   4, 2, 1),      # 8  -> 16
        nn.ConvTranspose2d(ngf, 1, 4, 2, 1),# 16 -> 32
        nn.Tanh(),
    )

def make_discriminator(ndf=64):
    return nn.Sequential(
        nn.Conv2d(1, ndf, 4, 2, 1, bias=False), nn.LeakyReLU(0.2, True),                                   # 32 -> 16
        nn.Conv2d(ndf, ndf*2, 4, 2, 1, bias=False), nn.BatchNorm2d(ndf*2), nn.LeakyReLU(0.2, True),        # 16 -> 8
        nn.Conv2d(ndf*2, ndf*4, 4, 2, 1, bias=False), nn.BatchNorm2d(ndf*4), nn.LeakyReLU(0.2, True),      # 8  -> 4
        nn.Conv2d(ndf*4, 1, 4, 1, 0, bias=False),                                                          # 4  -> 1
        nn.Flatten(),   # raw logits; BCEWithLogits used in the loss
    )

def init_weights(m):
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)): nn.init.normal_(m.weight, 0.0, 0.02)
    elif isinstance(m, nn.BatchNorm2d): nn.init.normal_(m.weight, 1.0, 0.02); nn.init.zeros_(m.bias)

for name, cfg in CONFIGS.items():
    g = make_generator(cfg["ngf"], cfg["g_bn"])
    print(f"{name:9s} generator params: {sum(p.numel() for p in g.parameters()):>9,}")
print(f"discriminator params: {sum(p.numel() for p in make_discriminator().parameters()):,}")

## 4. Metrics

Eyeballing a grid is not evidence. Two numbers per run:

- **Pairwise diversity**: mean L2 distance between random pairs of generated images, reported as a ratio to the same statistic on real data. Real = 1.0. Collapse drives this toward 0.
- **Class coverage / entropy**: a small CNN classifier is trained on real MNIST (~99% acc) and used to label generated samples. Reports how many of the 10 digits appear (>1% of samples) and the normalised entropy of the class histogram (1.0 = uniform, 0 = single class).

Caveat: the classifier labels garbage as *some* digit, so class coverage alone can overstate quality. Read it together with the grid and the diversity ratio.

In [ ]:
@torch.no_grad()
def pairwise_diversity(x, n_pairs=5000):
    x = x.flatten(1).float()
    i = torch.randint(0, len(x), (n_pairs,)); j = torch.randint(0, len(x), (n_pairs,))
    keep = i != j
    return (x[i[keep]] - x[j[keep]]).norm(dim=1).mean().item()

# real-data reference
real_ref = torch.cat([next(iter(loader))[0] for _ in range(8)])[:1000].to(device)
REAL_DIV = pairwise_diversity(real_ref)
print(f"real pairwise diversity: {REAL_DIV:.2f}")

class Clf(nn.Module):
    def __init__(s):
        super().__init__()
        s.c1 = nn.Conv2d(1, 32, 3, padding=1); s.c2 = nn.Conv2d(32, 64, 3, padding=1); s.fc = nn.Linear(64*8*8, 10)
    def forward(s, x):
        x = F.max_pool2d(F.relu(s.c1(x)), 2); x = F.max_pool2d(F.relu(s.c2(x)), 2)
        return s.fc(x.flatten(1))

def train_classifier():
    clf = Clf().to(device); opt = torch.optim.Adam(clf.parameters(), 1e-3)
    tf = T.Compose([T.Resize(IMG_SIZE), T.ToTensor(), T.Normalize((0.5,), (0.5,))])
    tr = torch.utils.data.DataLoader(torchvision.datasets.MNIST("./data", train=True, transform=tf), batch_size=256, shuffle=True)
    te = torch.utils.data.DataLoader(torchvision.datasets.MNIST("./data", train=False, transform=tf), batch_size=1000)
    for x, y in tr:
        x, y = x.to(device), y.to(device)
        opt.zero_grad(); F.cross_entropy(clf(x), y).backward(); opt.step()
    clf.eval(); correct = 0
    with torch.no_grad():
        for x, y in te: correct += (clf(x.to(device)).argmax(1).cpu() == y).sum().item()
    print(f"eval classifier test acc: {correct/10000:.4f}")
    return clf

clf = train_classifier()

@torch.no_grad()
def class_stats(x):
    p = F.softmax(clf(x), 1).argmax(1).cpu()
    hist = torch.bincount(p, minlength=10).float() / len(p)
    nz = hist[hist > 0]
    ent = -(nz * nz.log()).sum().item() / np.log(10)
    return dict(classes_covered=int((hist > 0.01).sum()), entropy=ent, hist=hist.numpy())

## 5. Training loop

In [ ]:
@torch.no_grad()
def sample(G, n=1000, z=None):
    G.eval()
    z = torch.randn(n, Z_DIM, 1, 1, device=device) if z is None else z
    out = G(z); G.train()
    return out

def train(name, cfg, epochs=EPOCHS):
    set_seed()
    G = make_generator(cfg["ngf"], cfg["g_bn"]).to(device); G.apply(init_weights)
    D = make_discriminator().to(device); D.apply(init_weights)
    oG = torch.optim.Adam(G.parameters(), lr=cfg["lr_g"], betas=(0.5, 0.999))
    oD = torch.optim.Adam(D.parameters(), lr=cfg["lr_d"], betas=(0.5, 0.999))
    bce = nn.BCEWithLogitsLoss()
    fixed_z = torch.randn(64, Z_DIM, 1, 1, device=device)
    hist = dict(lossD=[], lossG=[], div=[], entropy=[], D_real=[], D_fake=[])
    t0 = time.time()
    for ep in range(1, epochs + 1):
        lD = lG = dr = df = 0.0
        for real, _ in loader:
            real = real.to(device); b = real.size(0)
            z = torch.randn(b, Z_DIM, 1, 1, device=device)
            fake = G(z)
            # --- D step
            oD.zero_grad()
            out_r = D(real); out_f = D(fake.detach())
            loss_d = bce(out_r, torch.full_like(out_r, cfg["real_label"])) + bce(out_f, torch.zeros_like(out_f))
            loss_d.backward(); oD.step()
            # --- G step (non-saturating loss: G wants D(fake) -> 1; target is always a hard 1.0)
            oG.zero_grad()
            out_g = D(fake)
            loss_g = bce(out_g, torch.ones_like(out_g))
            loss_g.backward(); oG.step()
            lD += loss_d.item(); lG += loss_g.item()
            dr += torch.sigmoid(out_r).mean().item(); df += torch.sigmoid(out_f).mean().item()
        n = len(loader)
        xs = sample(G, 1000)
        div = pairwise_diversity(xs) / REAL_DIV
        ent = class_stats(xs)["entropy"]
        for k, v in zip(hist, [lD/n, lG/n, div, ent, dr/n, df/n]): hist[k].append(v)
        print(f"[{name}] ep {ep:2d}/{epochs} | lossD {lD/n:.3f} lossG {lG/n:.3f} | D(real) {dr/n:.2f} D(fake) {df/n:.2f} | div/real {div:.2f} | entropy {ent:.2f} | {time.time()-t0:.0f}s")
    return G, D, hist, fixed_z

def show_grid(G, fixed_z, title, path):
    imgs = sample(G, z=fixed_z).cpu()
    grid = make_grid(imgs, nrow=8, normalize=True, value_range=(-1, 1), padding=2)
    plt.figure(figsize=(7, 7)); plt.axis("off"); plt.title(title)
    plt.imshow(grid.permute(1, 2, 0)); plt.savefig(path, dpi=150, bbox_inches="tight"); plt.show()

### 5a. Baseline (healthy reference)

In [ ]:
G_base, D_base, H_base, z_base = train("baseline", CONFIGS["baseline"])
show_grid(G_base, z_base, "Baseline DCGAN", "baseline_output.png")

### 5b. Broken model: no BatchNorm in G + generator 8x narrower + lr=1e-2

In [ ]:
G_bad, D_bad, H_bad, z_bad = train("broken", CONFIGS["broken"])
show_grid(G_bad, z_bad, "BROKEN DCGAN (no BN, tiny G, lr=1e-2)", "collapsed_output.png")

**Check before moving on.** Look at `collapsed_output.png` and the `div/real` and `entropy` columns. If the grid is varied and `div/real` is above ~0.7, you did **not** produce collapse. The failure mode may instead be noise or blank output, which is a different problem (non-convergence) and not what the assignment asks about. If so, make the break harsher (e.g. `ngf=4`, `lr=2e-2`) and rerun this section.

### 5c. Fixed model: one technique applied

In [ ]:
print("Technique:", FIX, "->", CONFIGS["fixed"])
G_fix, D_fix, H_fix, z_fix = train("fixed", CONFIGS["fixed"])
show_grid(G_fix, z_fix, f"IMPROVED DCGAN ({FIX})", "improved_output.png")

## 6. Comparison

In [ ]:
runs = {"baseline": (G_base, H_base, z_base), "broken": (G_bad, H_bad, z_bad), f"fixed ({FIX})": (G_fix, H_fix, z_fix)}

# side-by-side grids, SAME latent vectors for every model
fig, axs = plt.subplots(1, 3, figsize=(18, 6.4))
for ax, (name, (G, H, z)) in zip(axs, runs.items()):
    ax.imshow(make_grid(sample(G, z=z_base).cpu(), nrow=8, normalize=True, value_range=(-1,1)).permute(1,2,0)); ax.set_title(name); ax.axis("off")
plt.tight_layout(); plt.savefig("comparison.png", dpi=150); plt.show()

# training curves
fig, axs = plt.subplots(1, 3, figsize=(18, 4))
for name, (G, H, z) in runs.items():
    axs[0].plot(H["div"], label=name); axs[1].plot(H["entropy"], label=name); axs[2].plot(H["lossG"], label=name)
axs[0].set_title("Pairwise diversity / real (1.0 = real data)"); axs[1].set_title("Class entropy (1.0 = uniform)"); axs[2].set_title("Generator loss")
for a in axs: a.set_xlabel("epoch"); a.legend()
plt.tight_layout(); plt.show()

# final metric table
print(f"{'run':22s} {'div/real':>9s} {'entropy':>8s} {'classes':>8s} {'D(real)':>8s} {'D(fake)':>8s}")
for name, (G, H, z) in runs.items():
    cs = class_stats(sample(G, 1000))
    print(f"{name:22s} {H['div'][-1]:9.2f} {cs['entropy']:8.2f} {cs['classes_covered']:>5d}/10 {H['D_real'][-1]:8.2f} {H['D_fake'][-1]:8.2f}")

# class histograms
fig, axs = plt.subplots(1, 3, figsize=(15, 3))
for ax, (name, (G, H, z)) in zip(axs, runs.items()):
    ax.bar(range(10), class_stats(sample(G, 1000))["hist"]); ax.set_title(name); ax.set_xticks(range(10)); ax.set_ylim(0, 1)
plt.tight_layout(); plt.show()

### Reading the numbers

- **Collapse signature:** low `div/real` and low entropy together, plus D(fake) near 0 while D(real) near 1 (D wins easily because G's outputs are all the same) or loss oscillating wildly.
- **Confound to address in your write-up if you used `FIX = "ttur"`:** the broken model had lr=1e-2 and the fixed one has lr 1e-4/4e-4. Any improvement could be attributed to *lowering* the LR rather than to *unequal* LRs. To isolate the effect, add a control run: `lr_g = lr_d = 2e-4` with the same no-BN tiny G. If the control matches the fixed run, unequal LRs contributed nothing and the real fix was simply a sane LR. Report that honestly rather than claiming TTUR did it.
- **If you used `label_smoothing`:** this is a clean single-variable ablation (only `real_label` differs). Expect a weaker effect at lr=1e-2. That is a legitimate finding.

## 7. Written explanation (fill in from YOUR results; the numbers above are the evidence)

**1. What modification caused the model to perform poorly?**
> The broken model combined three changes: (a) BatchNorm removed from the generator, (b) generator width reduced from 64 to 8, (c) learning rate raised to 1e-2 for both networks. *[State which you believe dominated and why. Without an ablation you cannot claim which one of the three was the cause.]*

**2. What did the generated images look like?**
> *[Describe `collapsed_output.png`: e.g. "most of the 64 samples were near-identical blobs resembling a single digit", and quote `div/real` and entropy from the table.]*

**3. Which technique did you use to improve the model?**
> *[e.g. "Different learning rates for G (1e-4) and D (4e-4), i.e. TTUR" or "one-sided label smoothing, real target 0.9".]*

**4. Did the solution improve the output? Why?**
> *[Compare `improved_output.png` against `collapsed_output.png` using `div/real`, entropy and classes covered. Mechanism: collapse happens when G finds one output D scores highly and D cannot punish it before G moves on. TTUR lets D track G's current output, so a single mode stops being a stable optimum. Label smoothing caps D's confidence, which keeps its gradients informative to G instead of saturating. State whether your control run supports this explanation.]*